# 📓 Text AI Module 1: LM Sampling, Temperature & Perplexity with Real LLMs
Welcome to Module 1 of Language Models! In this notebook, we replace abstract simulations with **a real pre-trained Large Language Model (Meta's/Qwen's 0.5B model)**.

We will explore:
1. **Real Next-Token Logits & Softmax Probability Distributions**
2. **Temperature-Scaled Sampling** on actual English vocabulary tokens
3. **Perplexity (PPL) Evaluation** on real coherent human sentences vs. gibberish text
4. **Interactive Text Generation** across greedy, balanced, and high-entropy decoding regimes

In [ ]:
# Install required packages if running in Colab/Jupyter
try:
    import transformers
except ImportError:
    !pip install -q transformers torch accelerate ipywidgets matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Dropdown, Text

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load small open-weights LLM (Qwen2.5-0.5B or Qwen3.5-0.8B)
MODEL_ID = "Qwen/Qwen3.5-0.8B"
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading tokenizer and model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)

model.eval()
print(f"Successfully loaded {MODEL_ID} on {device}!")

## 💡 1. Real Next-Token Logits & Softmax Probability Distributions
An autoregressive language model computes the joint probability of a sequence of tokens $x_{1:L}$ by predicting the next token $x_i$ given prior context $x_{1:i-1}$:

$$p(x_{1:L}) = \prod_{i=1}^{L} p(x_i \mid x_{1:i-1})$$

When we pass a prompt into our LLM, the final Transformer layer outputs unnormalized continuous scores called **logits** $z \in \mathbb{R}^{|\mathcal{V}|}$ across all vocabulary tokens.

Applying a **temperature** parameter $T > 0$ scales the logits before the Softmax transformation:

$$p(x_i = k \mid x_{1:i-1}) = \frac{\exp(z_k / T)}{\sum_{j \in \mathcal{V}} \exp(z_j / T)}$$

* **$T \to 0$ (Greedy Decoding):** Sharpens the distribution, selecting the top token with 100% certainty.
* **$T = 1.0$ (Unscaled Sampling):** Preserves the true, unscaled probability distribution.
* **$T > 1.0$ (High Entropy):** Flattens probabilities toward uniform randomness, giving low-ranked tokens a chance to be selected.

## 🎛️ 2. Interactive Real Logit Visualizer
Type any prompt below (or use the default) and adjust the **Temperature ($T$)** slider. Watch how the real top candidate tokens from the LLM's vocabulary dynamically scale their probabilities!

In [ ]:
@torch.no_grad()
def inspect_next_token_distribution(prompt="The capital of France is", temperature=1.0, top_k=8):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model(**inputs)
    
    # Extract logits for the last token position
    last_logits = outputs.logits[0, -1, :].cpu()  # Shape: [Vocab_Size]
    
    # Get top-K token candidates
    top_logits, top_indices = torch.topk(last_logits, top_k)
    top_tokens = [tokenizer.decode([idx]) for idx in top_indices]
    
    # Calculate temperature-scaled probabilities
    if temperature == 0.0:
        probs = torch.zeros_like(top_logits)
        probs[0] = 1.0
        sampled_token_idx = 0
    else:
        scaled_top_logits = top_logits / temperature
        probs = F.softmax(scaled_top_logits, dim=-1)
        sampled_token_idx = torch.multinomial(probs, num_samples=1).item()
        
    sampled_token = top_tokens[sampled_token_idx]
    
    clear_output(wait=True)
    print(f"Prompt: '{prompt}'")
    print(f"Temperature (T): {temperature:.2f}")
    print(f"Sampled Next Token: '{sampled_token}' (Index in Top-{top_k}: {sampled_token_idx})")
    
    # Visualizing probabilities
    fig, ax = plt.subplots(figsize=(10, 4))
    display_labels = [f"'{t}'" for t in top_tokens]
    colors = ['#ff7f0e' if i == sampled_token_idx else '#1f77b4' for i in range(top_k)]
    
    bars = ax.bar(display_labels, probs.numpy(), color=colors, alpha=0.85, edgecolor='black')
    ax.set_ylim(0, 1.08)
    ax.set_title(f"Real Top-{top_k} Next-Token Probabilities (Temperature T = {temperature:.2f})", fontsize=12, fontweight='bold')
    ax.set_xlabel("Candidate Vocabulary Tokens", fontsize=10)
    ax.set_ylabel("Probability $p(x_i)$")
    ax.grid(True, linestyle='--', alpha=0.4)
    
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.3f}", ha='center', va='bottom', fontsize=9)
        
    plt.tight_layout()
    plt.show()

interact(inspect_next_token_distribution,
         prompt=Text(value="The capital of France is", description="Prompt:"),
         temperature=FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description="Temp (T):"));

## 📈 3. Real Cross-Entropy Loss & Perplexity (PPL)
To measure how well an LLM models natural human language, we calculate its **Perplexity (PPL)**.

### Mathematical Definition
The Cross-Entropy loss $H$ averaged over sequence length $L$ is:

$$H = -\frac{1}{L} \sum_{i=1}^{L} \log p(x_i^{gt} \mid x_{1:i-1})$$

Perplexity is the exponential of the average loss:

$$\text{PPL} = \exp(H) = \exp\left( -\frac{1}{L} \sum_{i=1}^{L} \log p(x_i^{gt} \mid x_{1:i-1}) \right)$$

### Didactic Insight
* **Coherent Human Sentences:** The model assigns high probability to expected word transitions, resulting in low cross-entropy loss and **low Perplexity (~10–30)**.
* **Random / Nonsensical Text:** The model is highly 'surprised' at every step, yielding high loss and **exponentially large Perplexity (>500)**.

In [ ]:
@torch.no_grad()
def calculate_real_perplexity(text_sequence):
    encodings = tokenizer(text_sequence, return_tensors="pt").to(device)
    input_ids = encodings.input_ids
    
    if input_ids.shape[1] < 2:
        return 0.0, 0.0
        
    # Forward pass to get logits
    outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss  # PyTorch CrossEntropyLoss on shifted targets
    ppl = torch.exp(loss)
    
    return loss.item(), ppl.item()

# Compare coherent natural text vs. random gibberish
sample_fluent = "Artificial intelligence and language models predict the next token based on context."
sample_gibberish = "Banana submarine blue computer gravity jump frog bicycle university."

loss_fluent, ppl_fluent = calculate_real_perplexity(sample_fluent)
loss_gibberish, ppl_gibberish = calculate_real_perplexity(sample_gibberish)

print(f"=== Real LLM ({MODEL_ID}) Perplexity Benchmark ===\n")
print(f"1. Fluent Text   : '{sample_fluent}'")
print(f"   -> Cross-Entropy Loss: {loss_fluent:.4f} | Perplexity (PPL): {ppl_fluent:.2f}\n")
print(f"2. Gibberish Text: '{sample_gibberish}'")
print(f"   -> Cross-Entropy Loss: {loss_gibberish:.4f} | Perplexity (PPL): {ppl_gibberish:.2f}")

## 🧪 4. Interactive Real Text Generation Lab
Test how different decoding strategies affect real completions generated by our LLM!

* **Greedy ($T=0.0$):** Deterministic and repetitive; always picks the top logit.
* **Balanced ($T=0.7, \text{top\_p}=0.9$):** Standard natural generation with controlled sampling.
* **Creative / Chaotic ($T=1.8$):** High diversity, higher chance of hallucinations or unexpected word choices.

In [ ]:
@torch.no_grad()
def generate_interactive(prompt="In a distant galaxy, scientists discovered", regime="Balanced (T=0.7)", max_tokens=30):
    regime_configs = {
        "Greedy (T=0.0)": {"do_sample": False, "temperature": None},
        "Balanced (T=0.7)": {"do_sample": True, "temperature": 0.7, "top_p": 0.9},
        "Creative / Chaotic (T=1.8)": {"do_sample": True, "temperature": 1.8, "top_k": 50}
    }
    
    config = regime_configs[regime]
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        pad_token_id=tokenizer.eos_token_id,
        **config
    )
    
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    clear_output(wait=True)
    print(f"LLM Engine: {MODEL_ID}")
    print(f"Decoding Regime: {regime}")
    print("-" * 60)
    print(f"Full Output:\n{generated_text}")
    print("-" * 60)

interact(generate_interactive,
         prompt=Text(value="In a distant galaxy, scientists discovered", description="Prompt:"),
         regime=Dropdown(options=["Greedy (T=0.0)", "Balanced (T=0.7)", "Creative / Chaotic (T=1.8)"],
                         value="Balanced (T=0.7)",
                         description="Regime:"),
         max_tokens=widgets.IntSlider(value=35, min=10, max=100, step=5, description="Max Tokens:"));